In [13]:
import os
import json

import numpy as np
import pandas as pd

In [14]:
runs = [
    os.path.join('experiments', str(x)) for x in range(1616, 1656)
]

assert len(runs) == 40, len(runs)

In [15]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [16]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    if os.path.exists(os.path.join(run, 'init_time.txt')):
        init_time = float(open(os.path.join(run, 'init_time.txt')).read())
    else:
        init_time = 0.
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we first reach 5% relative error
    try:
        first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
        info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item() + init_time
    except:
        info['time_to_5p_rel_l2_error'] = np.nan
    
    df.loc[len(df)] = info

In [17]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to min. RL2E. (s)  \
Solution        w/ BC                                                
Hopf fibration  False            38.6 ± 0.4            165.5 ± 7.9   
                True             92.7 ± 4.2           582.8 ± 10.3   
Plane waves     False            85.2 ± 4.2           477.1 ± 46.8   
                True            100.9 ± 5.1           551.2 ± 14.3   
Radial waves    False           159.5 ± 5.4           577.7 ± 10.2   
                True                    NaN            596.2 ± 1.4   
Random solution False           63.7 ± 11.4           446.6 ± 78.4   
                True                    NaN            596.4 ± 2.2   

                            Min. RL2E. (%)  
Solution        w/ BC                       
Hopf fibration  False  1.13e-02 ± 3.02e-04  
                True   1.62e-02 ± 6.11e-04  
Plane waves     False  1.18e-02 ± 6.97e-04  
                True   1.13e-02 ± 6.57e-04  
Radial waves    False  1.91e-02 ± 6.87e-04  
                True   5.43e-02 ± 8.07e-04  
Random solution False  1.21e-02 ± 4.35e-04  
                True   5.68e-02 ± 1.47e-03

In [18]:
print(display_results.to_latex())

\begin{tabular}{lllll}
\toprule
 &  & Time to <5% RL2E. (s) & Time to min. RL2E. (s) & Min. RL2E. (%) \\
Solution & w/ BC &  &  &  \\
\midrule
\multirow[t]{2}{*}{Hopf fibration} & False & 38.6 ± 0.4 & 165.5 ± 7.9 & 1.13e-02 ± 3.02e-04 \\
 & True & 92.7 ± 4.2 & 582.8 ± 10.3 & 1.62e-02 ± 6.11e-04 \\
\cline{1-5}
\multirow[t]{2}{*}{Plane waves} & False & 85.2 ± 4.2 & 477.1 ± 46.8 & 1.18e-02 ± 6.97e-04 \\
 & True & 100.9 ± 5.1 & 551.2 ± 14.3 & 1.13e-02 ± 6.57e-04 \\
\cline{1-5}
\multirow[t]{2}{*}{Radial waves} & False & 159.5 ± 5.4 & 577.7 ± 10.2 & 1.91e-02 ± 6.87e-04 \\
 & True & NaN & 596.2 ± 1.4 & 5.43e-02 ± 8.07e-04 \\
\cline{1-5}
\multirow[t]{2}{*}{Random solution} & False & 63.7 ± 11.4 & 446.6 ± 78.4 & 1.21e-02 ± 4.35e-04 \\
 & True & NaN & 596.4 ± 2.2 & 5.68e-02 ± 1.47e-03 \\
\cline{1-5}
\bottomrule
\end{tabular}

